# Unidad 20. Estadística descriptiva, agrupación y ordenación

**Duración orientativa:** 7 horas · **Versión:** 1.0 · **UT3**

Una tabla puede contener demasiadas filas para comprenderla de un vistazo. Pandas permite resumirla, agrupar observaciones comparables y ordenar resultados. Estas operaciones simplifican, pero también ocultan detalles: aprenderás a interpretarlas con prudencia.

## Resultados de aprendizaje

Al finalizar podrás calcular resúmenes descriptivos, contar frecuencias, agrupar con groupby, aplicar varias agregaciones, ordenar resultados y explicar qué representa cada fila de una tabla agregada.

**Material:** datos/pandas/20_lecturas_estaciones_limpias.csv. Es la versión didáctica limpia de las lecturas de estaciones.

## 1. Resumir no es explicar

Media, mediana, mínimo, máximo y desviación típica describen un conjunto de valores. No demuestran causas ni sustituyen la revisión de los datos originales. La media es sensible a valores extremos; la mediana representa el valor central ordenado. Conviene comparar ambas cuando sospeches asimetrías.

Las frecuencias indican cuántas observaciones hay en cada categoría. Son esenciales antes de comparar grupos: una media calculada con dos observaciones tiene menos estabilidad que otra basada en cientos.

In [ ]:
from pathlib import Path
import pandas as pd

ruta = Path('..') / 'datos' / 'pandas' / '20_lecturas_estaciones_limpias.csv'
if not ruta.exists():
    ruta = Path('datos') / 'pandas' / '20_lecturas_estaciones_limpias.csv'
lecturas = pd.read_csv(ruta)
lecturas.head()

In [ ]:
print(lecturas[['temperatura_c', 'humedad_pct', 'pm25_ug_m3']].describe())
print('\nMedia PM2.5:', lecturas['pm25_ug_m3'].mean())
print('Mediana PM2.5:', lecturas['pm25_ug_m3'].median())
print('\nFrecuencia por zona:')
print(lecturas['zona'].value_counts())

## 2. Agrupar con groupby

groupby separa las filas en grupos definidos por una o más variables, aplica una operación a cada grupo y combina los resultados. La regla mental es dividir, aplicar y combinar. Tras agrupar por zona, una fila del resultado ya no representa una lectura individual: representa todas las lecturas de una zona.

Una agregación puede ser mean, median, min, max, count, size o una combinación de varias. count ignora ausencias en la columna contada; size cuenta filas. Como este archivo está limpio, coincidirán en muchas situaciones, pero no siempre ocurre así.

In [ ]:
pm25_por_zona = lecturas.groupby('zona', as_index=False)['pm25_ug_m3'].mean()
pm25_por_zona = pm25_por_zona.rename(columns={'pm25_ug_m3': 'pm25_media_ug_m3'})
pm25_por_zona

In [ ]:
resumen_zona = (
    lecturas.groupby('zona', as_index=False)
    .agg(
        lecturas=('id_lectura', 'size'),
        temperatura_media_c=('temperatura_c', 'mean'),
        pm25_media_ug_m3=('pm25_ug_m3', 'mean'),
        pm25_maxima_ug_m3=('pm25_ug_m3', 'max'),
    )
)
resumen_zona

### Agrupar por más de una variable

Añadir estacion_id permite separar zonas y estaciones. Es útil cuando una variable puede esconder diferencias internas. No añadas todas las columnas disponibles: cada dimensión adicional aumenta el número de grupos y puede dejar grupos con muy pocos datos. La pregunta debe decidir el nivel de detalle.

In [ ]:
resumen_estacion = (
    lecturas.groupby(['zona', 'estacion_id'], as_index=False)
    .agg(lecturas=('id_lectura', 'size'), pm25_media_ug_m3=('pm25_ug_m3', 'mean'))
)
resumen_estacion

## 3. Ordenar para comparar

sort_values ordena filas por una o varias columnas. ascending=False coloca primero los valores altos. El orden no cambia los datos ni los convierte en una conclusión: solo facilita comparar. Si hay empates, añade una segunda clave para que el criterio sea explícito.

Ordenar el resultado agregado suele ser más útil que ordenar la tabla original, porque responde directamente a preguntas como qué grupo tiene la media más alta.

In [ ]:
resumen_ordenado = resumen_zona.sort_values(
    by=['pm25_media_ug_m3', 'zona'], ascending=[False, True]
)
resumen_ordenado

## 4. Filtrar después de agregar

Una condición puede aplicarse antes o después de agrupar, pero responde a preguntas distintas. Filtrar lecturas con PM2.5 alta antes de groupby pregunta por esas lecturas; filtrar zonas cuya media sea alta pregunta por grupos completos. Mantén estas dos ideas separadas.

La siguiente regla usa 20 como umbral didáctico, no como recomendación sanitaria.

In [ ]:
zonas_media_alta = resumen_zona.loc[resumen_zona['pm25_media_ug_m3'] >= 20]
zonas_media_alta

## Errores frecuentes

- Interpretar la media como el valor de cada fila.
- Olvidar el tamaño de cada grupo antes de compararlo.
- Agrupar por una columna que no responde a la pregunta.
- Aplicar el filtro antes de agrupar cuando se quería filtrar el resumen.
- Perder nombres de columnas tras agg; usa nombres explícitos que indiquen medida y unidad.
- Concluir causalidad a partir de diferencias entre grupos.

## Práctica guiada

1. Compara media y mediana de temperatura y PM2.5.
2. Cuenta cuántas lecturas tiene cada estación.
3. Explica qué representa una fila de resumen_zona.
4. Ordena las estaciones por PM2.5 media de mayor a menor.
5. Cambia el umbral de zonas_media_alta y explica por qué cambia el resultado.

## Ejercicios

**Ejercicio 20.01.** Calcula mínimo, máximo, media y mediana de humedad_pct. Explica qué estadístico usarías para representar un valor central y por qué.

**Ejercicio 20.02.** Obtén las frecuencias de zona y estacion_id. Comprueba si cada estación pertenece a una única zona.

**Ejercicio 20.03.** Agrupa por estacion_id y calcula número de lecturas, temperatura media y PM2.5 máxima. Nombra claramente las columnas de salida.

**Ejercicio 20.04.** Ordena el resumen anterior por PM2.5 máxima descendente y, en empate, por estacion_id ascendente.

**Ejercicio 20.05.** Crea una tabla con las zonas cuya PM2.5 media sea mayor que la media global. Explica por qué primero necesitas calcular la media global.

Resuelve cada actividad en una celda nueva antes de consultar el cuaderno de soluciones.

In [ ]:
df_20 = pd.read_csv(Path('../datos/pandas/20_lecturas_estaciones_limpias.csv'))

In [ ]:
# Ejercicio 20.01
print("Minimo: ", df_20['humedad_pct'].min())
print("Maximo: ", df_20['humedad_pct'].max())
print(f"Media:  {df_20['humedad_pct'].mean():.2f}")
print("Mediana: ", df_20['humedad_pct'].median())

# Para representar un valor central utilizaría la mediana ya que escoge el valor central de los datos y la mediana hace un media de todos los valores. Si hubiese algun valor erroneo de temperatura, la media presentaria un resultado distorsionado

Ejercicio 20.02. Obtén las frecuencias de zona y estacion_id. Comprueba si cada estación pertenece a una única zona.

In [ ]:
print(f"Frecuencias de zona: {df_20['zona'].value_counts()}")
print(f"\nFrecuencias de zona: {df_20['estacion_id'].value_counts()}")
print(f"")

In [ ]:
# Ejercicio 20.03. Agrupa por estacion_id y calcula número de lecturas, temperatura media y PM2.5 máxima. Nombra claramente las columnas de salida.
resumen_estacion_id = (df_20.groupby('estacion_id', as_index=False)
.agg(
    lecturas=('id_lectura', 'size'),
    temperatura_media_c= ('temperatura_c', 'mean'),
    pm25_maxima=('pm25_ug_m3', 'max')
))
resumen_estacion_id


In [ ]:
#Ejercicio 20.04. Ordena el resumen anterior por PM2.5 máxima descendente y, en empate, por estacion_id ascendente.
resumen_ordenado = resumen_estacion_id.sort_values(
    by=['pm25_maxima', 'estacion_id'], ascending=[False, True]
)
resumen_ordenado

In [ ]:
zonas_media_alta = resumen_zona.loc[resumen_zona['pm25_media_ug_m3'] >= 20]

In [24]:
# Ejercicio 20.05. Crea una tabla con las zonas cuya PM2.5 media sea mayor que la media global. Explica por qué primero necesitas calcular la media global.
media_global = df_20['pm25_ug_m3'].mean()
media_por_zona = df_20.groupby('zona').agg(
    media_pm25 = ('pm25_ug_m3', 'mean')
)
df_nueva = media_por_zona[media_por_zona['media_pm25'] > media_global]
df_nueva
# Para poder comparar la media por zonas necesitas primero la media global de todas las zonas

,media_pm25
zona,
Centro,22.95


## Recursos

- Agrupación: https://pandas.pydata.org/docs/user_guide/groupby.html
- DataFrame.agg: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html
- Ordenación: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html